# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'GC_rec_1ks_2-50keV'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
# DATASET: str = "reconstructed"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, 2, 1)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A]["reconstructed"], E_min=E_min, E_max=E_max)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B]["reconstructed"], E_min=E_min, E_max=E_max)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(
        log, catalogue, sdl, camera,
    )
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
            # 'DCounts [sigma]': (cts - true_cts) / np.sqrt(true_cts),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing SCOX1:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing SCOX1:   0%|          | 0/23 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing LEMX-CAM1BS3: 100%|██████████| 23/23 [00:04<00:00,  4.81it/s]


In [17]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.2f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/texTable_Unit_results_NEWMODEL.tex',
    overwrite=True,
    **KWS,
)

In [16]:
tab

'\\begin{table*}\n\\begin{tabular}{lccccccr}\n\\hline\nSource & DthetaX_A & DthetaY_A & Dcts_A & DthetaX_B & DthetaY_B & Dcts_B & SNR \\\\\n\\hline\nSCOX1 & 0.09 & 0.75 & 68.72 & 0.16 & 0.82 & 106.56 & 987.80 \\\\\nGX5-1 & -0.00 & 1.11 & 13.62 & -0.05 & 0.32 & 16.35 & 106.69 \\\\\nGX349+2 & 0.05 & -3.43 & 27.05 & 0.01 & 0.11 & -14.54 & 70.15 \\\\\nGX9+1 & 0.14 & 3.43 & 29.26 & 0.02 & 1.20 & 16.87 & 60.57 \\\\\nGX17+2 & 0.26 & -2.83 & 3.44 & -0.01 & -4.64 & -30.79 & 58.46 \\\\\nGX340+0 & -0.21 & 6.51 & -0.21 & -0.01 & 1.43 & 12.04 & 42.88 \\\\\nGX13+1 & 0.29 & -1.37 & 41.76 & -0.04 & -1.91 & 38.98 & 38.20 \\\\\nX1820-303 & -0.06 & 4.72 & 31.89 & -0.19 & -4.24 & 15.14 & 32.45 \\\\\nGX3+1 & 0.03 & 0.09 & 23.05 & 0.23 & 3.97 & -3.55 & 31.73 \\\\\nCIRX1 & -0.69 & 8.23 & 11.52 & 0.26 & 5.11 & 15.51 & 31.14 \\\\\nGROJ1655-40 & -0.18 & -1.05 & 13.54 & 0.13 & 5.37 & -17.93 & 26.70 \\\\\nGX9+9 & 0.16 & -8.83 & -8.25 & 0.11 & 4.62 & -1.01 & 25.29 \\\\\nX1735-444 & 0.10 & -5.96 & 18.67 & -0.08 & -